# Session 9 — Multi-Head Quantum Attention

Objective:
Extend the trainable Quantum Transformer by introducing multiple quantum attention heads and evaluate whether combining multiple attention representations improves classification performance.

In [1]:
import numpy as np

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector

In [2]:
X, y = make_moons(
    n_samples=400,
    noise=0.15,
    random_state=42
)

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

Train Shape: (320, 2)
Test Shape : (80, 2)


## Multi-Head Quantum Attention Circuits

In [3]:
x_params = ParameterVector("x", 2)

# Head 1

q1_theta = ParameterVector("q1", 2)
k1_theta = ParameterVector("k1", 2)
v1_theta = ParameterVector("v1", 2)

# Head 2

q2_theta = ParameterVector("q2", 2)
k2_theta = ParameterVector("k2", 2)
v2_theta = ParameterVector("v2", 2)


def create_qkv_circuit(weight_params):

    qc = QuantumCircuit(2)

    qc.ry(x_params[0], 0)
    qc.ry(x_params[1], 1)

    qc.ry(weight_params[0], 0)
    qc.ry(weight_params[1], 1)

    qc.cx(0, 1)

    return qc


query_circuit_1 = create_qkv_circuit(q1_theta)
key_circuit_1 = create_qkv_circuit(k1_theta)
value_circuit_1 = create_qkv_circuit(v1_theta)

query_circuit_2 = create_qkv_circuit(q2_theta)
key_circuit_2 = create_qkv_circuit(k2_theta)
value_circuit_2 = create_qkv_circuit(v2_theta)

print("Multi-Head Attention Circuits Ready")

Multi-Head Attention Circuits Ready


## Initialising Parameters for Both Attention Heads

In [4]:
q1_weights = np.random.uniform(
    0,
    np.pi,
    2
)

k1_weights = np.random.uniform(
    0,
    np.pi,
    2
)

v1_weights = np.random.uniform(
    0,
    np.pi,
    2
)

q2_weights = np.random.uniform(
    0,
    np.pi,
    2
)

k2_weights = np.random.uniform(
    0,
    np.pi,
    2
)

v2_weights = np.random.uniform(
    0,
    np.pi,
    2
)

print("Q1 Weights:", q1_weights)
print("K1 Weights:", k1_weights)
print("V1 Weights:", v1_weights)

print("\nQ2 Weights:", q2_weights)
print("K2 Weights:", k2_weights)
print("V2 Weights:", v2_weights)

Q1 Weights: [2.73498059 1.87081911]
K1 Weights: [0.08790668 1.95890597]
V1 Weights: [0.20385387 1.71741971]

Q2 Weights: [1.79987651 1.73658425]
K2 Weights: [2.97806775 1.51412821]
V2 Weights: [1.23585685 1.54165955]


## Multi-Head Quantum Attention Embedding

In [5]:
def multihead_transformer_embedding(sample):

    # HEAD 1

    q1_state = Statevector.from_instruction(
        query_circuit_1.assign_parameters({
            x_params[0]: sample[0],
            x_params[1]: sample[1],
            q1_theta[0]: q1_weights[0],
            q1_theta[1]: q1_weights[1]
        })
    )

    k1_state = Statevector.from_instruction(
        key_circuit_1.assign_parameters({
            x_params[0]: sample[0],
            x_params[1]: sample[1],
            k1_theta[0]: k1_weights[0],
            k1_theta[1]: k1_weights[1]
        })
    )

    v1_state = Statevector.from_instruction(
        value_circuit_1.assign_parameters({
            x_params[0]: sample[0],
            x_params[1]: sample[1],
            v1_theta[0]: v1_weights[0],
            v1_theta[1]: v1_weights[1]
        })
    )

    attention1 = np.abs(
        np.vdot(
            q1_state.data,
            k1_state.data
        )
    )

    head1 = attention1 * np.abs(v1_state.data)

    # HEAD 2

    q2_state = Statevector.from_instruction(
        query_circuit_2.assign_parameters({
            x_params[0]: sample[0],
            x_params[1]: sample[1],
            q2_theta[0]: q2_weights[0],
            q2_theta[1]: q2_weights[1]
        })
    )

    k2_state = Statevector.from_instruction(
        key_circuit_2.assign_parameters({
            x_params[0]: sample[0],
            x_params[1]: sample[1],
            k2_theta[0]: k2_weights[0],
            k2_theta[1]: k2_weights[1]
        })
    )

    v2_state = Statevector.from_instruction(
        value_circuit_2.assign_parameters({
            x_params[0]: sample[0],
            x_params[1]: sample[1],
            v2_theta[0]: v2_weights[0],
            v2_theta[1]: v2_weights[1]
        })
    )

    attention2 = np.abs(
        np.vdot(
            q2_state.data,
            k2_state.data
        )
    )

    head2 = attention2 * np.abs(v2_state.data)

    return np.concatenate(
        [
            head1,
            head2
        ]
    )

## Multi-Head Transformer Embeddings

In [6]:
X_train_multihead = np.array(
    [
        multihead_transformer_embedding(sample)
        for sample in X_train
    ]
)

X_test_multihead = np.array(
    [
        multihead_transformer_embedding(sample)
        for sample in X_test
    ]
)

print("Train Shape:", X_train_multihead.shape)
print("Test Shape :", X_test_multihead.shape)

Train Shape: (320, 8)
Test Shape : (80, 8)


## Training Classification Head

In [7]:
classifier = LogisticRegression()

classifier.fit(
    X_train_multihead,
    y_train
)

print("Classifier Trained")

Classifier Trained


In [8]:
test_accuracy = classifier.score(
    X_test_multihead,
    y_test
)

print("Multi-Head Quantum Transformer Accuracy:")
print(test_accuracy)

Multi-Head Quantum Transformer Accuracy:
0.825


In [9]:
session7_accuracy = 0.825
session8_accuracy = 0.8375
session9_accuracy = test_accuracy

print("Session 7 Accuracy :", session7_accuracy)
print("Session 8 Accuracy :", session8_accuracy)
print("Session 9 Accuracy :", session9_accuracy)

print("\nSession 9 - Session 8:")
print(session9_accuracy - session8_accuracy)

Session 7 Accuracy : 0.825
Session 8 Accuracy : 0.8375
Session 9 Accuracy : 0.825

Session 9 - Session 8:
-0.012500000000000067
